# BuildFail-SLM: 02 - Source-Specific Cleaning (Stratified High-Value Extraction)

Cleans all 6 raw sources individually with **stratified category targeting** to ensure critical, high-impact categories (Docker, Configuration, Dependency, Test, and Code Diff repairs) are heavily represented:
1. **CI-REPAIR-BENCH**: 100% of real developer diffs & failure logs (565 rows)
2. **CI/CD Self-Healing (Main)**: Standardized categories, root causes, and fix commands (7,777 rows with 1,375+ Docker & Container errors)
3. **CI/CD Self-Healing (OOD)**: Strictly isolated out-of-distribution benchmark (489 rows)
4. **LogChunks**: Multi-lingual failure chunks across 28+ languages (797 rows)
5. **Java OSS Travis**: Java build failure patterns and categories (123 rows)
6. **GHALogs (Stratified Target 31,000)**: Quota-directed extraction targeting Docker/Containers (7,000), Tests (7,500), Dependencies (6,500), and Config (5,500)
7. **Travis CI (Target 20,000)**: Filtered build and test failure records across multiple languages

In [6]:
import os
import json
import gzip
import csv
import tarfile
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/01_raw")
CLEAN_DIR = Path("../data/02_source_clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data source:   {RAW_DIR.resolve()}")
print(f"Clean destination: {CLEAN_DIR.resolve()}")

Raw data source:   C:\Users\reddy\OneDrive\document\SLM fine tuning\data\01_raw
Clean destination: C:\Users\reddy\OneDrive\document\SLM fine tuning\data\02_source_clean


## 1. Clean CI-REPAIR-BENCH (Primary Code-Diff Supervision)

In [7]:
def clean_ci_repair_bench():
    ci_path = RAW_DIR / "ci_repair_bench/ci_repair_dataset.parquet"
    out_dir = CLEAN_DIR / "ci_repair_bench"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "ci_repair_clean.parquet"
    
    print(f"Processing CI-REPAIR-BENCH from {ci_path}...")
    df_raw = pd.read_parquet(ci_path)
    records = []
    
    for _, row in df_raw.iterrows():
        log_text = ""
        logs = row.get("logs")
        if hasattr(logs, "__iter__") and not isinstance(logs, (str, bytes)):
            for item in logs:
                if isinstance(item, dict) and item.get("log"):
                    log_text = str(item.get("log"))
                    break
        elif isinstance(logs, str):
            log_text = logs
        
        changed_files = row.get("changed_files")
        files_list = [str(x) for x in changed_files if x is not None] if hasattr(changed_files, "__iter__") and not isinstance(changed_files, str) else []
        first_file = files_list[0] if files_list else None
        diff_text = str(row.get("diff") or "").strip()
        
        records.append({
            "record_id": f"ci_repair_{row.get('id')}",
            "source_dataset": "ci_repair_bench",
            "source_record_id": str(row.get('id')),
            "repository": f"{row.get('repo_owner')}/{row.get('repo_name')}",
            "language": "Python",
            "ci_platform": "GitHub Actions",
            "workflow_name": str(row.get("workflow_name") or "CI"),
            "failure_log": log_text[:10000],
            "failure_chunk": log_text[-2000:] if len(log_text) > 2000 else log_text,
            "failure_type": "Code Error",
            "root_cause": f"Build failure repaired in commit {row.get('sha_success')}",
            "source_file": first_file,
            "actual_diff": diff_text,
            "fix_command": "",
            "verification_command": "",
            "has_log": bool(log_text.strip()),
            "has_diff": bool(diff_text),
            "repair_supervision_available": bool(diff_text)
        })
    
    df_clean = pd.DataFrame(records)
    df_clean.to_parquet(out_file, index=False)
    print(f"-> Saved CI-Repair clean: {len(df_clean)} records to {out_file}")
    return df_clean

df_ci_clean = clean_ci_repair_bench()

Processing CI-REPAIR-BENCH from ..\data\01_raw\ci_repair_bench\ci_repair_dataset.parquet...
-> Saved CI-Repair clean: 565 records to ..\data\02_source_clean\ci_repair_bench\ci_repair_clean.parquet


## 2. Clean CI/CD Self-Healing (Main: 7,777 rows & OOD Isolated: 489 rows)

In [8]:
def clean_cicd_self_healing():
    sh_dir = RAW_DIR / "cicd_self_healing"
    out_dir = CLEAN_DIR / "cicd_self_healing"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    main_csv = sh_dir / "main/cicd_pipeline_failures_dataset.csv"
    df_main_raw = pd.read_csv(main_csv)
    records_main = []
    for _, r in df_main_raw.iterrows():
        err = str(r.get("error_message") or "").strip()
        records_main.append({
            "record_id": f"cicd_sh_{r['id']}",
            "source_dataset": "cicd_self_healing",
            "source_record_id": str(r['id']),
            "repository": str(r.get("component") or "pipeline"),
            "language": str(r.get("runtime") or "Unknown").capitalize(),
            "ci_platform": "CI/CD Pipeline",
            "workflow_name": "Pipeline",
            "failure_log": err,
            "failure_chunk": err,
            "failure_type": str(r.get("category") or "Unknown"),
            "root_cause": str(r.get("root_cause") or ""),
            "source_file": "",
            "actual_diff": "",
            "fix_command": str(r.get("fix_command") or ""),
            "verification_command": str(r.get("verification_command") or ""),
            "has_log": bool(err),
            "has_diff": False,
            "repair_supervision_available": False
        })
    df_main_clean = pd.DataFrame(records_main)
    df_main_clean.to_parquet(out_dir / "cicd_selfhealing_clean.parquet", index=False)
    
    ood_csv = sh_dir / "ood/cicd_real_ood_world_dataset.csv"
    df_ood_raw = pd.read_csv(ood_csv)
    records_ood = []
    for _, r in df_ood_raw.iterrows():
        err = str(r.get("error_message") or "").strip()
        records_ood.append({
            "record_id": f"cicd_ood_{r['id']}",
            "source_dataset": "cicd_self_healing_ood",
            "source_record_id": str(r['id']),
            "repository": str(r.get("component") or "pipeline"),
            "language": str(r.get("runtime") or "Unknown").capitalize(),
            "ci_platform": "CI/CD Pipeline",
            "workflow_name": "Pipeline",
            "failure_log": err,
            "failure_chunk": err,
            "failure_type": str(r.get("category") or "Unknown"),
            "root_cause": str(r.get("root_cause") or ""),
            "source_file": "",
            "actual_diff": "",
            "fix_command": str(r.get("fix_command") or ""),
            "verification_command": str(r.get("verification_command") or ""),
            "has_log": bool(err),
            "has_diff": False,
            "repair_supervision_available": False
        })
    df_ood_clean = pd.DataFrame(records_ood)
    df_ood_clean.to_parquet(out_dir / "cicd_ood_clean.parquet", index=False)
    print(f"-> Saved Self-Healing Main: {len(df_main_clean)} rows, OOD Isolated: {len(df_ood_clean)} rows.")
    return df_main_clean, df_ood_clean

df_main_clean, df_ood_clean = clean_cicd_self_healing()

-> Saved Self-Healing Main: 7777 rows, OOD Isolated: 489 rows.


## 3. Clean LogChunks (797 rows) & Java Travis (123 rows)

In [9]:
def clean_logchunks():
    out_dir = CLEAN_DIR / "logchunks"
    out_dir.mkdir(parents=True, exist_ok=True)
    xml_files = list((RAW_DIR / "logchunks").rglob("*.xml"))
    records = []
    for xml_path in xml_files:
        lang = xml_path.parent.name
        repo_name = xml_path.stem.replace("@", "/")
        try:
            tree = ET.parse(xml_path)
            for i, ex in enumerate(tree.getroot()):
                log_elem = ex.find("Log")
                chunk_elem = ex.find("Chunk")
                cat_elem = ex.find("Category")
                kw_elem = ex.find("Keywords")
                chunk_txt = chunk_elem.text.strip() if chunk_elem is not None and chunk_elem.text else ""
                log_rel = log_elem.text.strip() if log_elem is not None and log_elem.text else ""
                cat_txt = cat_elem.text.strip() if cat_elem is not None and cat_elem.text else "Build Error"
                kw_txt = kw_elem.text.strip() if kw_elem is not None and kw_elem.text else ""
                if chunk_txt:
                    records.append({
                        "record_id": f"logchunk_{lang}_{xml_path.stem}_{i}",
                        "source_dataset": "logchunks",
                        "source_record_id": f"{xml_path.stem}_{i}",
                        "repository": repo_name,
                        "language": lang,
                        "ci_platform": "Travis CI",
                        "workflow_name": "Build",
                        "failure_log": f"Log: {log_rel}\nKeywords: {kw_txt}\nChunk:\n{chunk_txt}",
                        "failure_chunk": chunk_txt[:2000],
                        "failure_type": cat_txt,
                        "root_cause": f"Build failure detected in {repo_name} ({lang}) with keywords: {kw_txt}",
                        "source_file": "",
                        "actual_diff": "",
                        "fix_command": "",
                        "verification_command": "",
                        "has_log": True,
                        "has_diff": False,
                        "repair_supervision_available": False
                    })
        except Exception:
            continue
    df_lc = pd.DataFrame(records)
    out_file = out_dir / "logchunks_clean.parquet"
    df_lc.to_parquet(out_file, index=False)
    print(f"-> Saved LogChunks clean: {len(df_lc)} rows to {out_file}")
    return df_lc

def clean_java_travis():
    tar_path = RAW_DIR / "java_travis/logcat-categories-and-patterns.tar.gz"
    out_dir = CLEAN_DIR / "java_travis"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "java_travis_clean.parquet"
    records = []
    if tar_path.exists():
        with tarfile.open(tar_path) as tar:
            for member in tar.getmembers():
                if member.name.endswith(".json") and not member.name.startswith("."):
                    f = tar.extractfile(member)
                    if f:
                        try:
                            data = json.load(f)
                            repo_name = member.name.split("/")[-1].replace(".json", "")
                            for cat, details in data.items():
                                patterns = details.get("patterns", []) if isinstance(details, dict) else []
                                records.append({
                                    "record_id": f"java_tr_{repo_name}_{cat[:20]}",
                                    "source_dataset": "java_travis",
                                    "source_record_id": f"{repo_name}_{cat}",
                                    "repository": repo_name,
                                    "language": "Java",
                                    "ci_platform": "Travis CI",
                                    "workflow_name": "Maven/Gradle Build",
                                    "failure_log": f"Java build failure under category '{cat}': {patterns[:3]}",
                                    "failure_chunk": f"Pattern match: {patterns[0] if patterns else cat}",
                                    "failure_type": cat,
                                    "root_cause": f"Build failed in Java Travis CI under category {cat}",
                                    "source_file": "",
                                    "actual_diff": "",
                                    "fix_command": "",
                                    "verification_command": "",
                                    "has_log": True,
                                    "has_diff": False,
                                    "repair_supervision_available": False
                                })
                        except Exception:
                            continue
    df_jt = pd.DataFrame(records)
    df_jt.to_parquet(out_file, index=False)
    print(f"-> Saved Java Travis clean: {len(df_jt)} rows to {out_file}")
    return df_jt

df_lc = clean_logchunks()
df_jt = clean_java_travis()

-> Saved LogChunks clean: 797 rows to ..\data\02_source_clean\logchunks\logchunks_clean.parquet
-> Saved Java Travis clean: 123 rows to ..\data\02_source_clean\java_travis\java_travis_clean.parquet


## 4. Stratified Extraction: GHALogs (Target 31,000) & Travis (Target 20,000)

In [10]:
def clean_ghalogs_and_travis_stratified():
    # 1. Stratified GHALogs Extraction
    gha_path = RAW_DIR / "ghalogs/runs.json.gz"
    out_gha = CLEAN_DIR / "ghalogs"
    out_gha.mkdir(parents=True, exist_ok=True)
    
    quotas = {'docker': 7000, 'test': 7500, 'dep': 6500, 'config': 5500, 'general': 4500}
    counts = {'docker': 0, 'test': 0, 'dep': 0, 'config': 0, 'general': 0}
    gha_records = []
    
    print("Stratifying GHALogs extraction targeting Docker, Tests, Dependencies & Config...")
    with gzip.open(gha_path, "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if sum(counts.values()) >= sum(quotas.values()): break
            try:
                r = json.loads(line)
                meta = r.get("metadata", {})
                conclusion = meta.get("conclusion")
                if conclusion in ["failure", "cancelled", "timed_out"]:
                    text = f"{r.get('repository_name', '')} {meta.get('name', '')} {meta.get('display_title', '')} {r.get('workflow_path', '')}".lower()
                    repo_name = r.get("repository_name") or meta.get("head_repository", {}).get("full_name") or "github/repo"
                    wf_name = meta.get("name") or r.get("workflow_path") or "CI Workflow"
                    disp_title = meta.get("display_title", "")
                    
                    target_cat = None
                    if any(k in text for k in ['docker', 'container', 'image', 'compose', 'dockerfile', 'containerd']) and counts['docker'] < quotas['docker']:
                        target_cat = 'Docker / Container Error'
                        counts['docker'] += 1
                    elif any(k in text for k in ['test', 'pytest', 'junit', 'check', 'lint', 'spec']) and counts['test'] < quotas['test']:
                        target_cat = 'Test Failure'
                        counts['test'] += 1
                    elif any(k in text for k in ['pip', 'npm', 'yarn', 'maven', 'dep', 'install', 'requirements', 'package']) and counts['dep'] < quotas['dep']:
                        target_cat = 'Dependency Error'
                        counts['dep'] += 1
                    elif any(k in text for k in ['yaml', 'yml', 'config', 'workflow error', 'actions/checkout', 'syntax']) and counts['config'] < quotas['config']:
                        target_cat = 'Configuration Error'
                        counts['config'] += 1
                    elif counts['general'] < quotas['general']:
                        target_cat = 'Build Error'
                        counts['general'] += 1
                        
                    if target_cat:
                        gha_records.append({
                            "record_id": f"ghalogs_{meta.get('id', len(gha_records))}_{len(gha_records)}",
                            "source_dataset": "ghalogs",
                            "source_record_id": str(meta.get('id') or r.get('_id')),
                            "repository": repo_name,
                            "language": "Multi",
                            "ci_platform": "GitHub Actions",
                            "workflow_name": wf_name,
                            "failure_log": f"GitHub Actions workflow '{wf_name}' failed at event {meta.get('event', 'push')} for branch {meta.get('head_branch', 'main')}. Title: {disp_title}",
                            "failure_chunk": f"Workflow run #{meta.get('run_number')} failed with conclusion: {conclusion}. Title: {disp_title}",
                            "failure_type": target_cat,
                            "root_cause": f"GitHub Actions {target_cat} in workflow '{wf_name}'",
                            "source_file": "",
                            "actual_diff": "",
                            "fix_command": "",
                            "verification_command": "",
                            "has_log": True,
                            "has_diff": False,
                            "repair_supervision_available": False
                        })
            except Exception:
                continue
    df_gha = pd.DataFrame(gha_records)
    df_gha.to_parquet(out_gha / "ghalogs_clean.parquet", index=False)
    print(f"-> Saved Stratified GHALogs: {len(df_gha)} rows (Docker: {counts['docker']}, Test: {counts['test']}, Dep: {counts['dep']}, Config: {counts['config']})")
    
    # 2. Travis CI (Target 20,000)
    tr_path = RAW_DIR / "travis/19314170/final-2017-01-25.csv.gz"
    out_tr = CLEAN_DIR / "travis"
    out_tr.mkdir(parents=True, exist_ok=True)
    tr_records = []
    count = 0
    print("Extracting 20,000 multi-lingual builds from TravisTorrent...")
    for chunk in pd.read_csv(tr_path, compression="gzip", chunksize=25000, low_memory=False):
        failed = chunk[chunk["tr_status"].isin(["failed", "errored"])]
        for _, row in failed.iterrows():
            status = row.get('tr_status')
            tr_records.append({
                "record_id": f"travis_{row.get('tr_build_id')}_{count}",
                "source_dataset": "travis",
                "source_record_id": str(row.get('tr_build_id')),
                "repository": str(row.get("gh_project_name") or "travis/project"),
                "language": str(row.get("gh_lang") or "Unknown").capitalize(),
                "ci_platform": "Travis CI",
                "workflow_name": "Travis Build",
                "failure_log": f"Travis CI build {row.get('tr_build_id')} failed for project {row.get('gh_project_name')} with status {status}",
                "failure_chunk": f"Build status: {status}, tests ran: {row.get('tr_tests_ran', 0)}, tests failed: {row.get('tr_tests_failed', 0)}",
                "failure_type": "Test Failure" if status == 'failed' else "Build Error",
                "root_cause": f"Build failed with status {status}",
                "source_file": "",
                "actual_diff": "",
                "fix_command": "",
                "verification_command": "",
                "has_log": True,
                "has_diff": False,
                "repair_supervision_available": False
            })
            count += 1
            if count >= 20000: break
        if count >= 20000: break
    df_tr = pd.DataFrame(tr_records)
    df_tr.to_parquet(out_tr / "travis_clean.parquet", index=False)
    print(f"-> Saved Travis clean: {len(df_tr)} rows to {out_tr / 'travis_clean.parquet'}")
    return df_gha, df_tr

df_gha, df_tr = clean_ghalogs_and_travis_stratified()

Stratifying GHALogs extraction targeting Docker, Tests, Dependencies & Config...
-> Saved Stratified GHALogs: 30053 rows (Docker: 6053, Test: 7500, Dep: 6500, Config: 5500)
Extracting 20,000 multi-lingual builds from TravisTorrent...
-> Saved Travis clean: 20000 rows to ..\data\02_source_clean\travis\travis_clean.parquet
